<a href="https://colab.research.google.com/github/ddickson28/FPSO-BN/blob/C_child%2C-P_child/FPSOBN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install mbnpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires nump

In [4]:
#Import modules
import numpy as np #importing a module
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy
import itertools
from scipy.stats import truncnorm
from math import erf, sqrt

In [5]:
#1 Define function for calculating combinations
def generate_indexed_combinations(state_counts):

    # Create ranges for each variable: 0..n_states-1
    ranges = [range(n) for n in state_counts]

    # Compute Cartesian product
    combos = list(itertools.product(*ranges))

    # Convert to numpy array
    return np.array(combos, dtype=int)

#Example usage:
#state_counts = [3, 2, 3]  # Child: 3 states, Parent1: 2 states, Parent2: 3 states
#C_matrix = generate_indexed_combinations(state_counts)
#print(C_matrix)

In [6]:
"Defining the child node probability table requires defining all of the parent state
"combinations and then using a T-Normal distribution to create discrete bins
"across the defined child states|parent combination

# Define further node weights as required, ensure consistent ordering and sum to
# 1
N1 = 0.7
N2 = 0.3

#Define the numeric scores for the states of ea. node. Follows ordering of nodes
#above.

P_Repair=np.array([1,0])
P_Offload=np.array([1, 0.5, 0])
#P_Additional=np.array([1, 0.75, 0.5, 0.25, 0])

#Define the weighted scores for the node vectors
A = N1*P_Repair
B = N2*P_Offload

print(A)
print(B)

#Create pairwise sum w/ specific ordering. Vector 1 moves slowest, vector 2
#moves fastest. ***Read this code block, format to extend?***

Parent_combination = A[:, None] + B[None, :]

print(Parent_combination)

#Takes two rows and flattens into a 1-D vector
mu = Parent_combination.reshape(-1)

print(mu)

#Calculating truncated normal from mu vector above and specficied variance

var = 0.0225
std = np.sqrt(var)

#Define intervals for truncnorm 0, 1/3 2/3, 1. Consider automating for more
#states. Reflects Low, Med, High
bins = np.array([0.0, 0.33, 0.66, 1])

#Build truncnorm and calculate CDF for each bin interval

a = (0 - mu) / std
b = (1 - mu) / std

cdf_val = truncnorm.cdf(bins[:, None], a, b, loc=mu, scale=std)

prob_vectors = np.diff(cdf_val, axis=0).T
prob_vectors = prob_vectors[:,::-1]
prob_vectors.sum(axis=1)

print(prob_vectors)

#***Need to build the order here, 1st from each column***
flat_prob = prob_vectors.T.flatten()

print(flat_prob)


[0.7 0. ]
[0.3  0.15 0.  ]
[[1.   0.85 0.7 ]
 [0.3  0.15 0.  ]]
[1.   0.85 0.7  0.3  0.15 0.  ]
[[9.76589404e-01 2.34026514e-02 7.94475104e-06]
 [8.78008098e-01 1.21678748e-01 3.13153661e-04]
 [5.95945713e-01 3.97078238e-01 6.97604834e-03]
 [8.38681938e-03 4.22147308e-01 5.69465873e-01]
 [4.00456519e-04 1.36368287e-01 8.63231256e-01]
 [1.08250616e-05 2.77960699e-02 9.72193105e-01]]
[9.76589404e-01 8.78008098e-01 5.95945713e-01 8.38681938e-03
 4.00456519e-04 1.08250616e-05 2.34026514e-02 1.21678748e-01
 3.97078238e-01 4.22147308e-01 1.36368287e-01 2.77960699e-02
 7.94475104e-06 3.13153661e-04 6.97604834e-03 5.69465873e-01
 8.63231256e-01 9.72193105e-01]


In [7]:
"Building the BN and relationship"
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy

#1 Define the variables (nodes). two states false/true
#At this stage the nodes are not connected in any way.

PreviousRepair = variable.Variable('PreviousRepair', ['True','False']) #Variable Class inside the variable module, creates an objected called "PreviousRepair".
OffloadCycle = variable.Variable('OffloadCycles', ['High','Med','Low']) #Variable states are organised worst to best. e.g. previous repair True and offload cycles High is worst
CrackLocationInterest = variable.Variable('CrackLocationInterest', ['High','Med','Low'])


#2 Define the cpm of the two variables from 1.

cpm_PreviousRepair = cpm.Cpm(
									[PreviousRepair], no_child=1,
									 C=np.array([[0],[1]], dtype=int),
									 p=np.array([0.5,0.5])
)

cpm_OffloadCycle = cpm.Cpm(
									[OffloadCycle], no_child=1,
									 C=np.array([[0],[1],[2]], dtype=int),
									 p=np.array([0.33,0.33,0.34])
)

# Creates all possible child states given parents
state_counts = [3, 2, 3]  # Child: 3 states, Parent1: 2 states, Parent2: 3 states
C_child= generate_indexed_combinations(state_counts)

P_child=flat_prob

cpm_CrackLocationInterest = cpm.Cpm(
									[CrackLocationInterest, OffloadCycle, PreviousRepair], no_child=1,
									 C=C_child,
									 p=P_child
)

print(cpm_CrackLocationInterest)

<CPM representing P(CrackLocationInterest | OffloadCycles, PreviousRepair) at 0x7b61f6d7a870>
+-------------------------+-----------------+------------------+-------------+
|   CrackLocationInterest [   OffloadCycles |   PreviousRepair ]           p |
+=========================+=================+==================+=============+
|                       0 [               0 |                0 ] 0.976589    |
+-------------------------+-----------------+------------------+-------------+
|                       0 [               0 |                1 ] 0.878008    |
+-------------------------+-----------------+------------------+-------------+
|                       0 [               0 |                2 ] 0.595946    |
+-------------------------+-----------------+------------------+-------------+
|                       0 [               1 |                0 ] 0.00838682  |
+-------------------------+-----------------+------------------+-------------+
|            :            [        : 